# 📚 Notebook 1: Literature Review & Theoretical Background
## Skin Cancer Detection Using ISIC 2024 Challenge Dataset

**Objective:** Establish the scientific context, review existing approaches, identify gaps, and position our methodology.

---

## 1.1 The Clinical Problem

Skin cancer is the most common cancer worldwide. **Melanoma**, while accounting for only ~1% of 
skin cancers, is responsible for the **majority of skin cancer deaths**. The critical factor is 
**early detection**:

| Stage at Diagnosis | 5-Year Survival Rate |
|---|---|
| Localized (Stage I/II) | **99%** |
| Regional (Stage III) | 68% |
| Distant (Stage IV) | **32%** |

*Source: American Cancer Society, Cancer Facts & Figures 2024*

The **ISIC 2024 Challenge** introduces a unique paradigm: combining **3D Total Body Photography 
(TBP)** with machine learning. TBP captures standardized whole-body images, from which individual 
lesion tiles and quantitative features are extracted. This enables:

1. **Standardized imaging** — reduced variability from lighting, camera, and technique
2. **Contextual analysis** — comparing a lesion against the patient's other lesions
3. **Rich metadata** — 40+ quantitative features per lesion (color, shape, texture, symmetry)

### Why This Matters
Traditional dermoscopy-based ML relies solely on close-up lesion images. The 3D-TBP paradigm 
provides structured metadata that encodes **dermatologist-level clinical features** — features that 
are typically assessed qualitatively during clinical examination but are now available as quantitative 
measurements.

## 1.2 Evolution of Computational Approaches

### 1.2.1 CNN-Based Methods (2016–2021)

The landmark paper by **Esteva et al. (2017, Nature)** demonstrated that a fine-tuned **Inception-v3** 
could match Board-certified dermatologists at skin lesion classification. This catalyzed a wave of 
CNN-based approaches:

| Model | Key Innovation | ISIC Performance |
|---|---|---|
| Inception-v3 (Esteva 2017) | First human-competitive result | AUC ~0.91 |
| ResNet-50 (2018 ISIC) | Deeper features + transfer learning | AUC ~0.93 |
| EfficientNet-B4/B5 (2020) | Compound scaling for efficiency | AUC ~0.95 |
| SE-ResNeXt (2019 ISIC winner) | Squeeze-excitation attention | AUC ~0.94 |

**Strengths:** Strong visual pattern recognition, good transfer learning from ImageNet.

**Limitations:**
- 🔴 **Domain shift**: Performance degrades across different scanners and lighting conditions
- 🔴 **Class imbalance sensitivity**: Rare malignant cases are overwhelmed; high false-negative risk
- 🔴 **Data hungry**: Require large, curated, balanced datasets for optimal performance
- 🔴 **Black-box nature**: Limited interpretability for clinical adoption

### 1.2.2 Vision Transformers (2021–2023)

**ViT (Dosovitskiy et al., 2020)** brought self-attention to computer vision, theoretically 
capturing long-range spatial dependencies that CNNs miss. Variants explored for dermoscopy:

- **DeiT**: Data-efficient training with knowledge distillation
- **Swin Transformer**: Hierarchical feature maps via shifted windows
- **CoAtNet**: Hybrid CNN-Transformer combining local and global features

**Practical assessment:** In ISIC benchmarks, well-tuned Transformers showed **marginal improvement 
(1-2% AUC)** over EfficientNets, while requiring **3-5x more compute** and **larger datasets**. 
The added complexity is rarely justified for dermoscopic classification tasks.

### 1.2.3 Multi-Modal / Hybrid Approaches (2023–Present)

The most significant recent development is the recognition that **clinical metadata matters**:

- **Yao et al. (2023)**: Showed metadata-augmented CNNs outperform image-only models by 3-5% AUC
- **ISIC 2024 top solutions**: Multiple entries relied primarily on **gradient-boosted trees on 
  metadata**, with image models as secondary signals
- **Ha et al. (2024)**: Multi-modal fusion of dermoscopic images with clinical variables achieved 
  state-of-the-art on multiple benchmarks

## 1.3 Identified Gap: Underutilized Metadata Features

### The Problem

Most published approaches treat clinical metadata (age, sex, anatomical site) as **auxiliary inputs** — 
typically concatenated to CNN feature vectors before a final classification layer. This approach has 
three critical flaws:

1. **Information bottleneck**: Rich metadata is compressed into a few dimensions alongside hundreds of 
   CNN features, diluting its signal
2. **Modality mismatch**: CNNs are optimized for image gradients; tabular features have fundamentally 
   different statistical properties (categorical, skewed, multi-modal)
3. **Feature underutilization**: The ISIC 2024 dataset provides **40+ TBP-derived features** — far 
   richer than the 3-4 simple demographic variables typically used

### The ISIC 2024 Metadata Goldmine

The TBP system provides features that encode the **ABCDE clinical criteria** computationally:

| Clinical Criterion | TBP Feature(s) | What It Captures |
|---|---|---|
| **A**symmetry | `tbp_lv_symm_2axis` | Two-axis symmetry score |
| **B**order irregularity | `tbp_lv_norm_border`, `tbp_lv_eccentricity` | Edge regularity, shape |
| **C**olor variation | `tbp_lv_color_std_mean`, `tbp_lv_stdL` | Color heterogeneity |
| **D**iameter | `clin_size_long_diam_mm`, `tbp_lv_areaMM2` | Size measurements |
| **E**volution (proxy) | `tbp_lv_deltaL/A/B` | Lesion-vs-skin differences |

Additionally, `tbp_lv_nevi_confidence` and `tbp_lv_dnn_lesion_confidence` provide **pre-computed 
neural network assessments** — essentially distilled knowledge from prior models.

## 1.4 Our approach: phased metadata-first, then multi-modal (planned)

### Positioning

We position this project at the intersection of the identified gap:

```
Traditional image-only ──────────────────── Our phased pipeline
     CNN/ViT on images                      │
     Metadata = afterthought                ├── Track A: Feature-engineered metadata
                                            │   → LightGBM / XGBoost (primary)
                                            │
                                            ├── **Phase 2 (planned):** CNN on images + late fusion
                                            │   → Transfer learning (secondary)
                                            │
                                            └── Late Fusion: Weighted ensemble
                                                → Optimal weights via validation
```

### Why This Should Work

**Mathematical argument:** Let $X_t$ be tabular features and $X_i$ be image features. If the 
conditional mutual information $I(Y; X_i | X_t) > 0$, then images provide additional discriminative 
information beyond metadata. Conversely, $I(Y; X_t | X_i) > 0$ means metadata adds value beyond 
images. **Phase 2 fusion** will combine both:

$$P(Y|X_t, X_i) \propto P(Y|X_t)^{w_t} \cdot P(Y|X_i)^{w_i}$$

where $w_t, w_i$ are modality weights optimized on validation data.

**Practical argument:** Top Kaggle solutions for ISIC 2024 consistently showed that metadata-based 
models (LightGBM with engineered features) achieved 80-90% of the winning score *without any image 
data*. **Phase 2** will add images via late fusion for a complementary boost.

### Key Design Decisions

| Decision | Rationale |
|---|---|
| LightGBM as primary model | Handles tabular data optimally; naturally handles missing values, multicollinearity |
| Focal Loss for image CNN | Mathematical solution for extreme class imbalance (γ=2 down-weights easy negatives) |
| StratifiedGroupKFold CV | Preserves class ratio AND prevents patient data leakage across folds |
| pAUC as primary metric | Official ISIC 2024 metric; focuses on clinically relevant high-sensitivity region |
| Late fusion (not early) | Avoids modality dominance; allows independent optimization of each track |

## 1.5 Theoretical Framework

### 1.5.1 The Bias-Variance Perspective

Under extreme class imbalance (~1020:1), the **naive Bayes-optimal classifier** predicts all samples 
as benign (achieving 99.9% accuracy but 0% sensitivity). This is a **high-bias** solution.

Our strategy addresses this systematically:

| Component | Bias-Variance Effect |
|---|---|
| `scale_pos_weight = 1020` | Reduces bias toward majority class |
| Feature engineering | Reduces bias by providing richer representations |
| Ensemble (LightGBM + XGBoost) | Reduces variance through model averaging |
| Focal Loss (image model) | Reduces bias by re-weighting gradient contributions |
| Cross-validation | Provides unbiased performance estimates |

### 1.5.2 Assumptions and Mitigations

**IID Assumption**: Standard ML assumes training samples are independent and identically distributed.

- **Violation**: Multiple lesions from the same patient are NOT independent
- **Mitigation**: GroupKFold splitting by `patient_id` ensures no patient appears in both train and 
  validation sets

**Stationarity Assumption**: Feature distributions are constant across time/source.

- **Violation**: Different imaging centers may have systematic biases
- **Mitigation**: `attribution` column tracks image source; we can analyze and control for this

### 1.5.3 Why Gradient Boosting for Tabular Data

**No Free Lunch applied to our domain**: Grinsztajn et al. (2022, NeurIPS) demonstrated that 
tree-based models (XGBoost, LightGBM, Random Forest) **consistently outperform** deep learning on 
tabular data, particularly when:
- Features have heterogeneous types (continuous, categorical)
- Feature scales vary widely
- Dataset contains missing values
- Sample-to-feature ratio is moderate

All four conditions apply to our metadata. The mathematical advantage of trees: they learn 
**axis-aligned decision boundaries** that naturally capture **feature interactions** without explicit 
specification — critical when we don't know which feature pairs interact.

### 1.5.4 Focal Loss: Mathematical Derivation

Standard cross-entropy for class $y \in \{0, 1\}$ with prediction $p$:

$$\text{CE}(p, y) = -y\log(p) - (1-y)\log(1-p)$$

Focal Loss (Lin et al., 2017) adds a modulating factor $(1-p_t)^\gamma$:

$$\text{FL}(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$

where $p_t = p$ if $y=1$, else $p_t = 1-p$.

**Effect**: When $\gamma = 2$:
- Easy benign (correctly predicted, $p \approx 0$): $(1-0.9)^2 = 0.01$ → gradient reduced 100x
- Hard malignant (missed, $p \approx 0.3$): $(1-0.3)^2 = 0.49$ → gradient nearly full strength

This mathematically ensures the model **focuses learning capacity on the rare, difficult malignant cases**.

## 1.6 Summary

| Aspect | Our Position |
|---|---|
| **Gap identified** | Rich TBP metadata is underutilized; most work is image-only |
| **Primary model** | LightGBM on engineered tabular features |
| **Phase 2 image model** | EfficientNet-class CNN (planned) |
| **Fusion (Phase 2)** | Late fusion of tabular OOF + image OOF |
| **Imbalance handling** | Phase 1: scale_pos_weight + StratifiedGroupKFold; Phase 2: add focal / sampling for CNNs |
| **Primary metric** | Partial AUC (pAUC) above 88% TPR |
| **Key assumption** | TBP metadata + images together > either alone |

---

**Next Notebook:** `02_EDA.ipynb` — Load the full dataset and perform comprehensive exploration.
